# Ch7 Pre-training 实战 教案

**课程名称：** Pre-training 实战：炼制你的 TinyLlama

**预计总时长：** 90-100 分钟

**源文件：** `Ch7_Pretraining/Ch7_Pretraining.ipynb`（共 30 个 Cell，Cell 0-29）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-5 min | 开场 + 环境准备 | Cell 0-5 | 5 min |
| 5-15 min | 理论：预训练的数学本质与 Scaling Laws | Cell 2 | 10 min |
| 15-30 min | 数据工程：语料加载 + 字符词表 + 滑动窗口 Dataset | Cell 6-10 | 15 min |
| 30-35 min | **休息 + 回顾** | -- | 5 min |
| 35-50 min | 模型定义 + 训练前基线 | Cell 11-13 | 15 min |
| 50-70 min | 训练循环：LR 调度 + 训练主循环 + 可视化 | Cell 14-19 | 20 min |
| 70-75 min | **休息 + 回顾** | -- | 5 min |
| 75-85 min | 文本生成 + 温度实验 | Cell 20-23 | 10 min |
| 85-90 min | Checkpoint 管理 + 总结 | Cell 24-27 | 5 min |
| 90-100 min | 练习 + 讨论 | Cell 28-29 | 10 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 和 PyTorch 已安装（`import torch; print(torch.__version__)`）
- [ ] 确认 `data/` 目录下有中文语料文件
- [ ] 确认 `models/` 目录可写（Checkpoint 保存位置）
- [ ] 确认中文字体文件 `assets/fonts/NotoSansCJKsc-Regular.otf` 存在（Matplotlib 可视化需要）
- [ ] 如有 GPU，确认 CUDA 可用：`print(torch.cuda.is_available())`
- [ ] 预跑一遍全部 Cell，确认训练约 13 秒内完成（GPU），CPU 约 2-5 分钟
- [ ] 准备白板或屏幕画板，用于手绘滑动窗口示意图

---

## 第一段：开场与环境准备（Cell 0-5）

📍 运行 Cell 0-1（Markdown 导读）、Cell 3-4（环境准备代码）、Cell 5（预训练流程全景图）

⏱ 时间分配：5 分钟

🎯 本段目标
- 建立学习动机：为什么要从零做预训练？
- 确认环境就绪（PyTorch、中文字体、数据目录）
- 通过全景图让学生对预训练流程有全局感

🗣 讲课话术

> 大家好！前面几章我们从 Embedding 一路走到了 GPT 模型架构，也学了 Tokenizer。现在，我们手里有了所有的零件——词表、注意力机制、Transformer Block。今天我们要把这些零件**组装起来，真正开炉炼丹**。
>
> 这一章的核心问题就一句话：**如何让一个随机初始化的模型学会说人话？** 答案就是预训练——让模型在大量文本上反复练习预测下一个字。
>
> 我们先运行环境准备。Cell 4 会告诉我们用的设备。（运行 Cell 4）看到输出了吗？`Using device: cuda`，PyTorch 版本 2.10.0。如果大家是 CPU 也没关系，我们的 TinyLlama 只有 **85 万参数**，CPU 上 2-5 分钟就能训完。
>
> 现在运行 Cell 5，看这张全景图。（运行 Cell 5）大家看，预训练流程从左到右：**原始语料 -> 清洗过滤 -> Tokenize -> 构造训练对 -> 送入模型 -> 计算 Loss -> 反向传播**。底下写了四个核心要点，我念一下——数据质量大于数据数量、学习率调度至关重要、定期保存 checkpoint、验证集 loss 判断过拟合。这四点，今天每一个我们都会实操。

👀 输出要点
- Cell 4：`Using device: cuda`（或 cpu），`PyTorch version: 2.10.0+cu128`
- Cell 5：全景图可视化 + 4 条核心要点文字

❓ 预判问题
- **Q：预训练和微调的区别是什么？**
  A：预训练是在大量无标签文本上学习通用语言能力（学说话），微调是在特定任务数据上调整模型（学做事）。本章做预训练，Ch8-Ch10 做微调。
- **Q：为什么不直接用 Hugging Face 上的预训练模型？**
  A：工程中当然用现成的。但理解预训练原理能帮你诊断问题、选择合适的模型、甚至在特定领域做 continued pretraining。

➡️ 转场

> 好，环境就绪。现在让我们先花 10 分钟理解预训练背后的数学——为什么预测下一个词就能训出 GPT 这样的模型？

---

## 第二段：理论——预训练的数学本质与 Scaling Laws（Cell 2）

📍 浏览 Cell 2（长 Markdown，无需运行代码）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解 Next-Token Prediction = 最大似然估计 = 交叉熵损失
- 通过数值例子建立直觉：什么样的预测 loss 低？
- 了解 Scaling Laws 和 Chinchilla 法则的核心结论

🗣 讲课话术

> 预训练听起来很玄，但数学上其实很优雅。模型要做的就一件事：给定前文，预测下一个字的概率分布。我们来看 Cell 2 的公式。
>
> 想象你在做完形填空。句子是「今天天气真____」。你脑子里可能会给「好」50%的概率，「差」20%，「热」15%，「冷」10%。模型也一样，它输出一个概率分布。
>
> 训练目标就是让模型给正确答案的概率越高越好。数学上就是**最大化似然**，等价于**最小化交叉熵损失** $\mathcal{L} = -\frac{1}{T}\sum \log P(x_t | \text{前文})$。
>
> 来看 Cell 2 里的数值例子。「今天天气真好」这 6 个字，模型对「气」给了 0.35 的概率，loss 只有 1.05；但对第一个字「今」只给了 0.002，loss 高达 6.21。**越可预测的 token，loss 贡献越小。** 这意味着模型自动把注意力集中在难以预测的 token 上——这就是自监督学习的精妙之处。
>
> 再看 Scaling Laws。Kaplan 2020 年发现，**模型 loss 和参数量、数据量、计算量之间是幂律关系**。参数翻 10 倍，loss 降低约 1.19 倍。这就是为什么大家疯狂扩大模型。
>
> 但 2022 年 Chinchilla 论文说了一个关键洞察：**参数和数据要等比例增长**，最优训练 token 数约等于参数量的 20 倍。7B 的模型最优需要 140B tokens。所以不是模型越大越好，数据不够就是浪费算力。
>
> 大家记住两个关键数字：**Chinchilla 比例 20:1**，以及**随机初始化的 loss 等于 ln(vocab_size)**——我们马上就会验证这个。

👀 输出要点
- 本段无代码输出，重点是 Cell 2 Markdown 中的公式和表格
- 数值例子：「今天天气真好」的平均 loss = 2.46
- Chinchilla 法则：最优 token 数 ≈ 20 x 参数量

❓ 预判问题
- **Q：为什么用交叉熵而不是 MSE？**
  A：因为预测下一个 token 是分类问题（从 V 个候选中选一个），不是回归问题。交叉熵是分类问题的标准损失函数。
- **Q：Scaling Laws 是不是意味着参数越多越好？**
  A：在固定计算预算下，Chinchilla 表明参数和数据要平衡。比如 LLaMA-1 用了远超最优量的数据训练较小模型，这叫过度训练（over-training），实际部署时推理成本更低。

➡️ 转场

> 理论讲完了，下面动手！第一步是准备数据。巧妇难为无米之炊，我们先看看语料长什么样。

---

## 第三段：数据工程——语料加载 + 字符词表 + 滑动窗口 Dataset（Cell 6-10）

📍 运行 Cell 7（读取语料 + 统计）、Cell 8（构建字符词表 + 编解码测试）、Cell 10（TextDataset + DataLoader + 样本展示）

⏱ 时间分配：15 分钟（语料 4 分钟 + 词表 4 分钟 + 滑动窗口 7 分钟）

🎯 本段目标
- 了解中文预训练语料的基本特征
- 理解字符级词表的构建过程
- 掌握滑动窗口如何构造 (input, target) 训练对
- 理解 train/val 划分与 DataLoader 的作用

🗣 讲课话术

> 先运行 Cell 7，加载我们的中文语料。（运行 Cell 7）看输出——**总共 4,292 个字符，257 行，366 个唯一字符，其中 83.2% 是中文**。这是一个很小的语料，主要是 AI/深度学习领域的短句子。实际预训练当然需要上百 GB 的数据，但原理是一样的。
>
> 看前 500 字符预览：都是类似「深度学习是用多层神经网络学习数据表示的方法」这样的句子。大家注意，这些句子本身就是关于 AI 的知识——我们等于在教模型学 AI 教科书。
>
> 接下来运行 Cell 8，构建字符级词表。（运行 Cell 8）**vocab_size = 366**，很小对吧？因为我们用的是字符级编码，每个汉字就是一个 token。看编解码测试——「深度学习是用多层神」编码成 `[364, 238, 164, 145, 57, ...]` 再解码回来完全一致。
>
> 词表组成：86.3% 中文字符，10.4% 英文字母，2.5% 标点。和 Ch6 的 BPE Tokenizer 不同，字符级编码不会合并子词，好处是简单透明，坏处是序列长、信息密度低。
>
> 现在重点来了——**滑动窗口**。大家看 Cell 9 的 Markdown。（展示 Cell 9）打个比方：想象你在教小孩认字。你用一个 64 字长的窗户在整篇文章上滑动。每次你指着窗户里的字问小孩：看完这 64 个字，下一个字是什么？窗户每滑一格，就产生一个训练样本。
>
> 运行 Cell 10。（运行 Cell 10）看输出：**训练集 3,798 个样本，验证集 366 个样本**。每个 batch 是 32 个序列 x 64 tokens = **2,048 tokens/batch**。119 个 batch 就是一个 epoch。
>
> 看样本展示——输入是「准备应当制定应急预案……」，目标就是把输入右移一位：第一个变成「备」，完美对应 next-token prediction。

👀 输出要点
- Cell 7：总字符 4,292，257 行，366 唯一字符，中文占 83.2%
- Cell 8：vocab_size = 366，中文 316 个（86.3%），编解码测试通过
- Cell 10：训练集 3,798 样本，验证集 366 样本，batch shape [32, 64]
- Cell 10：训练集 batch 数 119

❓ 预判问题
- **Q：为什么用字符级而不用 BPE？**
  A：教学目的——字符级更直观，不需要训练 Tokenizer。实际 LLM 都用 BPE 或 SentencePiece，压缩比更高。Ch6 已经讲了 BPE。
- **Q：block_size = 64 是不是太小了？**
  A：GPT-2 用 1024，LLaMA 用 2048-4096。我们用 64 是因为语料只有 4K 字符，上下文窗口太大反而浪费。实际中 block_size 越大，模型能捕获的长距离依赖越多，但显存消耗也越大。
- **Q：训练集和验证集怎么划分的？**
  A：按 9:1 比例简单切分。前 90% 做训练，后 10% 做验证。预训练通常用时间或文档边界划分，不做 shuffle，避免数据泄露。

➡️ 转场

> 数据准备好了，下面该构建模型了。我们会复用上一章的 GPT 架构——不过今天的版本更迷你。

---

## 休息 + 回顾（第 30-35 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 预训练的目标是最小化交叉熵损失，本质就是最大似然估计——让模型尽量给正确的下一个 token 高概率。
2. 我们的中文语料只有 4,292 个字符、366 个唯一字符。字符级编码简单直观，编解码可逆。
3. 滑动窗口以 block_size=64 在文本上滑动，每滑一格生成一个 (input, target) 对，共 3,798 个训练样本。

**下一段预告：** 我们要定义一个只有 85 万参数的 mini GPT 模型，验证随机初始化时的理论 loss 值。

---

## 第四段：模型定义 + 训练前基线（Cell 11-13）

📍 运行 Cell 12（GPT 模型定义 + 参数量统计）、Cell 13（理论初始 loss + 训练前生成对比）

⏱ 时间分配：15 分钟（模型架构 7 分钟 + 训练前基线 8 分钟）

🎯 本段目标
- 理解 mini GPT 的配置参数和参数量分布
- 验证理论初始 loss = ln(vocab_size)
- 看到随机初始化模型生成的胡言乱语，建立训练前的基线

🗣 讲课话术

> 运行 Cell 12。（运行 Cell 12）看输出——**模型参数量 848,384，不到 85 万**。来看参数分布：
>
> - Token Embedding（wte）：46,848，占 5.5%——把 366 个字符映射到 128 维向量
> - Position Embedding（wpe）：8,192，占 1.0%——64 个位置 x 128 维
> - Attention（x4 层）：264,192，占 31.1%
> - MLP（x4 层）：526,848，占 **62.1%**——看到了吗？MLP 占了大头！
> - LM Head 和 wte 共享权重，不额外增加参数
>
> 打个比方：如果说 Attention 是模型的「眼睛」，负责看清上下文，那 MLP 就是模型的「大脑」，占了六成的脑容量。这和论文中的发现一致——MLP 存储了大量的事实知识。
>
> 现在运行 Cell 13，看训练前的基线。（运行 Cell 13）
>
> **理论初始 loss = ln(366) = 5.9026**。为什么？随机初始化时，模型对每个字符的预测概率大约是 1/366 = 0.0027。$-\log(0.0027) = 5.90$。一会儿训练时我们看实际初始 loss 是不是接近这个值。
>
> 看训练前的生成结果——Prompt 是「深度学习是用多层神经网络学习数据表示的方法。」，模型输出的是什么？「资合向有。限少其子有灰灰温的次建或向容证……」**完全是乱码**，毫无意义。这就是随机初始化的模型——每个字符都是随机挑的。我们训练完再回来对比。

👀 输出要点
- Cell 12：参数量 848,384；MLP 占 62.1%，Attention 占 31.1%
- Cell 12：config = vocab_size=366, block_size=64, n_layer=4, n_head=4, n_embd=128, dropout=0.1
- Cell 13：理论初始 loss = ln(366) = 5.9026
- Cell 13：训练前生成输出为乱码（随机字符拼接）

❓ 预判问题
- **Q：为什么 MLP 占参数量的比例最大？**
  A：每层 MLP 有两个线性层，中间维度是 n_embd 的 4 倍（128->512->128）。4 层就是 4x(128x512 + 512x128) = 524,288。MLP 是模型存储知识的主要组件。
- **Q：LM Head 和 wte 共享权重是什么意思？**
  A：输出层（把 128 维向量映射回 366 个字符的概率）直接复用 Embedding 层的权重矩阵（转置）。好处是减少参数、提高泛化，这是 GPT-2 以来的标准做法。
- **Q：4 层 4 头够吗？**
  A：对 4K 字符的小语料够了。GPT-2 Small 有 12 层 12 头，LLaMA-7B 有 32 层 32 头。层数和头数的选择取决于数据规模和计算预算。

➡️ 转场

> 模型定义好了，基线也记录了。下面进入最激动人心的环节——**训练！** 我们要看着 loss 从 5.9 一路下降。

---

## 第五段：训练循环——LR 调度 + 训练主循环 + 可视化（Cell 14-19）

📍 运行 Cell 16（学习率调度可视化）、Cell 17（训练配置）、Cell 18（训练主循环）、Cell 19（Loss/PPL 可视化）

⏱ 时间分配：20 分钟（LR 调度 5 分钟 + 训练 10 分钟 + 可视化分析 5 分钟）

🎯 本段目标
- 理解 Warmup + Cosine Decay 学习率调度的原理和作用
- 观察完整训练过程中 loss 的变化
- 理解过拟合的信号（train loss 下降但 val loss 上升）
- 建立 Perplexity（困惑度）的直觉

🗣 讲课话术

> 在开始训练之前，我们先聊一个关键话题——**学习率调度**。运行 Cell 16。（运行 Cell 16）
>
> 看这四张图：固定学习率、线性衰减、Cosine 衰减、**Warmup + Cosine Decay**。第四张是实际最常用的。
>
> 打个比方：学习率就像你骑自行车的速度。刚上路时你得慢慢加速（Warmup），因为路况不明；中间全速前进；快到终点时慢慢减速（Cosine Decay），精细调整到最佳位置。如果一上来就全速，可能摔跤（梯度爆炸）；如果到终点还不减速，可能冲过头（loss 震荡不收敛）。
>
> 运行 Cell 17 看训练配置。（运行 Cell 17）**2,000 次迭代，每次处理 2,048 tokens，总共处理 409 万 tokens**。Warmup 200 步（10%），学习率 3e-4。预估在 GPU 上不到 1 分钟。
>
> 好，运行 Cell 18！（运行 Cell 18，等待训练完成）
>
> 大家看训练日志——
> - **Iter 0：Train Loss = 5.9248，PPL = 374.2**。和我们算的理论值 5.9026 几乎一样！验证了随机初始化 loss 约等于 ln(V)。
> - **Iter 200：Train Loss 骤降到 1.6679，PPL = 5.3**。只用了 200 步，困惑度从 374 降到 5！意思是模型从「366 个字里瞎猜」进步到「5 个字里选一个」。
> - **Iter 400：Train Loss = 0.4582，PPL = 1.6**。模型已经基本背住了训练集。
> - 但注意 **Val Loss 从 Iter 200 的 4.47 到 Iter 400 的 3.88 就开始触底了**。之后 Train Loss 继续下降到 0.069，但 Val Loss 反弹到 4.48——典型的**过拟合信号**！
>
> **最佳验证 Loss 出现在 Iter 400 左右：Val Loss = 3.88，PPL = 48.5**。整个训练耗时 13.4 秒，吞吐约 30 万 tokens/sec。
>
> 运行 Cell 19 看可视化。（运行 Cell 19）这张图很说明问题：蓝线（Train Loss）一路下降趋近 0，橙线（Val Loss）先降后升形成一个 V 字底。**红色虚线**是理论随机 loss 5.90。大家看，两条线的初始值都在红线附近。
>
> Cell 19 底部输出：最终 Train Loss 0.069（PPL 1.1），Val Loss 4.48（PPL 88.4），**Loss 降低了 24.1%**。
>
> 为什么过拟合这么严重？因为我们只有 4,292 个字符的语料！这就像让你反复背同一页教科书——考试可能满分，但换一页就不行了。实际 LLM 用 TB 级数据就是为了避免这个问题。

👀 输出要点
- Cell 17：max_iters=2000, learning_rate=3e-4, warmup=200 步, 总处理 tokens=4,096,000
- Cell 18 训练日志：
  - Iter 0: Train Loss=5.9248, Val Loss=5.9210, PPL=374.2
  - Iter 200: Train Loss=1.6679, Val Loss=4.4703, PPL=5.3/87.4
  - Iter 400: Train Loss=0.4582, Val Loss=3.8813（**最佳**），PPL=1.6/48.5
  - Iter 1800: Train Loss=0.0690, Val Loss=4.4818, PPL=1.1/88.4
  - 总耗时 13.4s, 吞吐 305,237 tok/s
- Cell 19：Loss 降低 24.1%，可视化显示明显过拟合

❓ 预判问题
- **Q：为什么 Warmup 要占 10%？**
  A：经验值。开始时模型参数是随机的，梯度方向不稳定。Warmup 让优化器热身，逐步增大步长。GPT-3 论文用了约 0.1-0.6% 的 Warmup。我们的 10% 偏保守但对小数据集更安全。
- **Q：过拟合了怎么办？**
  A：三个方向——（1）增大数据集（根本解）；（2）增大 dropout（已设 0.1）；（3）减少训练步数，提前停止（early stopping），我们的 best model 就是保存的 Iter 400 附近的 checkpoint。
- **Q：PPL = 48.5 是好是坏？**
  A：对 4K 字符的小语料来说算合理。意思是模型平均在约 48 个字符中犹豫。GPT-2 在 WikiText-103 上的 PPL 约 20-30，但那是大模型大数据。

➡️ 转场

> 训练完了，最激动人心的时刻——让我们看看模型现在能说什么了！

---

## 休息 + 回顾（第 70-75 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. 我们的 mini GPT 只有 85 万参数（4 层 4 头，128 维），MLP 占了 62%，是存储知识的主力。
2. 训练初始 loss = 5.9248 约等于 ln(366) = 5.9026，完美验证理论；最佳 Val Loss = 3.88 出现在约 Iter 400，之后过拟合。
3. Warmup + Cosine Decay 是 LLM 标配的学习率策略：先慢后快再慢，像骑车先加速后匀速最后减速。

**下一段预告：** 我们要看训练前后的生成对比，还有一个有趣的温度实验。

---

## 第六段：文本生成 + 温度实验（Cell 20-23）

📍 运行 Cell 21（加载最佳模型 + 生成）、Cell 22（训练前后对比）、Cell 23（多 Prompt + 温度对比）

⏱ 时间分配：10 分钟

🎯 本段目标
- 直观感受训练前后生成质量的巨大差异
- 理解温度参数对生成多样性的影响
- 理解 top-k 采样的作用

🗣 讲课话术

> 运行 Cell 21，加载最佳模型并生成 200 个 token。（运行 Cell 21）
>
> 大家看生成的文本——「空注不同步。学习用梯度下文。DP继习监督学参数量……」。老实说，还是不太通顺对吧？但至少能看到**中文句子结构**，有标点、有动词、有名词。比训练前的完全随机好太多了。
>
> 运行 Cell 22 看训练前后对比。（运行 Cell 22）
>
> 大家对比看：
> - **训练前：** 「资合向有。限少其子有灰灰温的次建或向容证盖记力力结传漂l看失C……」——纯随机
> - **训练后：** 「过拟合。数据一致性。微调参数据模型可幻果。梯度与回滚策略……」——虽然语法不完美，但已经出现了「过拟合」「数据一致性」「微调」「梯度」这些**训练数据中的真实概念**！
>
> 模型在 85 万参数、4K 字符语料、13 秒训练的条件下，已经学会了一些基本的说话模式。想象一下，把参数扩大到 70 亿、语料扩大到万亿 token、训练几周——那就是 ChatGPT。
>
> 现在看 Cell 23 的**温度实验**。（运行 Cell 23）
>
> 三个温度对比：
> - **T=0.7（保守）：** 文本更连贯但容易重复——「学习用梯度……学习用指标……模型训练……模型训练……」
> - **T=1.0（标准）：** 更多样但偶尔跑偏——「用偏好。反向量是每个位置。下工程。」
> - **T=1.5（创意）：** 基本是乱码——「神赖。数可的最大立梯子学法数据生验证集学同子……」
>
> 温度的物理意义：**softmax 之前的 logits 除以 T**。T 小 -> 概率分布更尖锐 -> 总是选最可能的字 -> 保守重复。T 大 -> 概率分布更平 -> 低概率字也有机会被选 -> 多样但混乱。

👀 输出要点
- Cell 21：加载 best model（Val Loss=3.88），生成 200 tokens，可看到基本的中文句式
- Cell 22：训练前纯随机 vs 训练后出现真实概念词（过拟合、梯度、数据准备等）
- Cell 23：T=0.7 保守重复，T=1.0 平衡多样，T=1.5 接近乱码

❓ 预判问题
- **Q：为什么训练后的文本还是不通顺？**
  A：三个原因——（1）语料太小（4K 字符），模型见过的语言模式有限；（2）模型太小（85 万参数），表达能力不足；（3）字符级编码的序列太长，难以捕获长距离依赖。增大任何一个都能改善。
- **Q：实际产品中温度一般设多少？**
  A：对话类 0.7-0.9，代码生成 0.2-0.5，创意写作 0.9-1.2。还会配合 top-k（本例 k=40）或 top-p（nucleus sampling）一起用。

➡️ 转场

> 训练完了、生成也看了。最后一个工程实践——如何保存和加载 Checkpoint？

---

## 第七段：Checkpoint 管理 + 本章总结（Cell 24-27）

📍 运行 Cell 25（Checkpoint 保存/加载代码）、浏览 Cell 26-27（总结 Markdown）

⏱ 时间分配：5 分钟

🎯 本段目标
- 掌握完整 Checkpoint 的保存内容（模型 + 优化器 + epoch + loss + config）
- 理解 best model vs full checkpoint 的区别
- 回顾本章全流程

🗣 讲课话术

> 运行 Cell 25。（运行 Cell 25）看输出——Checkpoint 保存了 5 样东西：**model_state_dict（57 个参数张量）、optimizer_state_dict、epoch、loss、config**。文件大小 9.84 MB。
>
> 为什么要存优化器状态？因为 Adam 为每个参数维护了动量（一阶矩）和方差（二阶矩）的估计值。如果只存模型不存优化器，恢复训练时优化器要从头学习这些统计量，等于 Warmup 要重来。
>
> 对比 best model 文件只有 3.32 MB——因为它只存了 model_state_dict。部署时只需要这个小文件。
>
> 翻看 Cell 26 的总结图谱：**语料库 -> Tokenize -> 滑动窗口 -> DataLoader -> GPT 模型 -> Next-Token Loss -> 反向传播 -> 生成采样**。这就是完整的预训练闭环。
>
> Cell 27 提到下一章 Ch8 SFT——有了预训练好的基座模型后，我们要通过指令微调把它变成能对话的模型。那里会用到 Loss Masking 和 ChatML 格式。

👀 输出要点
- Cell 25：Checkpoint 9.84 MB（含 57 个参数张量 + 优化器状态），Best model 3.32 MB
- Cell 25：config = {vocab_size: 366, block_size: 64, n_layer: 4, n_head: 4, n_embd: 128, dropout: 0.1}

❓ 预判问题
- **Q：为什么 Checkpoint 比 best model 大 3 倍？**
  A：Adam 优化器为每个参数维护两个状态张量（一阶矩 m 和二阶矩 v），所以优化器状态约等于 2 倍的模型参数大小。加上模型本身，就是约 3 倍。
- **Q：实际训练中多久保存一次？**
  A：通常每 N 步保存一次（N 取决于训练时长），保留最近 K 个 + 最佳验证 loss 的 checkpoint。大模型训练会用分布式 checkpoint（如 FSDP/DeepSpeed），支持断点续训。

➡️ 转场

> 核心流程走完了。最后留几道思考题和练习，大家动手试试。

---

## 第八段：练习与讨论（Cell 28-29）

📍 浏览 Cell 28（Extra 思考题）、Cell 29（练习空间）

⏱ 时间分配：10 分钟

🎯 本段目标
- 学生独立思考和动手实验
- 加深对关键概念的理解

🗣 讲课话术

> Cell 28 列了 4 道思考题，我们挑两道讨论一下，剩下的课后完成。
>
> **第 1 题：为什么初始 Loss 约等于 ln(vocab_size)？** 这个我们已经验证过了。随机初始化时模型对每个 token 的预测概率约 1/V，$-\log(1/V) = \log(V) = \ln(366) = 5.90$，实测 5.9248，非常接近。
>
> **第 2 题：过拟合 vs 欠拟合。** 我们的训练就是一个活生生的例子！Train Loss 降到 0.069 但 Val Loss 升到 4.48。如果 Train Loss 也降不下去，那就是欠拟合——模型太小或学习率太低。
>
> 大家可以在 Cell 29 的练习空间里尝试：
> 1. 修改 `block_size` 为 32 或 128，重新训练，观察效果
> 2. 修改 `n_layer` 或 `n_embd`，看参数量和收敛速度的变化
> 3. 尝试降低 `max_iters` 到 400（在 Val Loss 最低点停止），看生成效果是否更好

**提示节奏**
- 0-2 分钟：自己思考和尝试
- 2 分钟：提示——改 config 字典中的参数即可，改完需要重新创建模型和优化器
- 4 分钟：关键代码——`config['block_size'] = 32` 后重新运行 Cell 12 和 Cell 17-18

**常见错误**
- 忘记重新创建 Dataset（block_size 变了，TextDataset 也要重新创建）
- 忘记重新创建优化器（模型参数变了，优化器绑定的参数引用也要更新）
- block_size 设太大超过语料长度，导致样本数为 0

**验证标准**
- 初始 Loss 仍然约等于 ln(vocab_size)
- 不同配置的收敛速度有可观察的差异
- 能说出过拟合发生在哪一步

❓ 预判问题
- **Q：第 3 题梯度裁剪是什么？**
  A：`clip_grad_norm_` 在反向传播后、优化器 step 前，把梯度的 L2 范数裁剪到阈值内（通常 1.0）。防止单步更新太大导致训练崩溃。我们的代码中实际已经用了 `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`。
- **Q：第 4 题 OOV 怎么处理？**
  A：字符级编码中 OOV 概率低（只要字符在训练文本中出现过就有）。BPE 通过 byte-level fallback 彻底解决 OOV——Ch6 讲过。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，运行环境准备 + 全景图 | 0-5 |
| 5 | 理论：MLE、交叉熵、Scaling Laws | 2 |
| 15 | 数据工程：语料 + 词表 + 滑动窗口 | 6-10 |
| 30 | **休息** | -- |
| 35 | 模型定义 + 训练前基线 | 11-13 |
| 50 | LR 调度 + 训练主循环 + 可视化 | 14-19 |
| 70 | **休息** | -- |
| 75 | 文本生成 + 温度实验 | 20-23 |
| 85 | Checkpoint + 总结 | 24-27 |
| 90 | 练习 + 讨论 | 28-29 |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)$$

$$\text{Perplexity} = e^{\mathcal{L}}$$

### 数据规模速查

| 指标 | 值 |
|:---|:---|
| 总字符数 | 4,292 |
| 唯一字符（vocab_size） | 366 |
| 中文字符占比 | 83.2%（316 个） |
| block_size | 64 |
| 训练样本数 | 3,798 |
| 验证样本数 | 366 |
| batch_size | 32 |
| tokens/batch | 2,048 |

### 模型配置速查

| 参数 | 值 |
|:---|:---|
| n_layer | 4 |
| n_head | 4 |
| n_embd | 128 |
| dropout | 0.1 |
| 总参数量 | 848,384 |
| MLP 占比 | 62.1% |
| Attention 占比 | 31.1% |

### 训练关键数值

| 指标 | 值 |
|:---|:---|
| 理论初始 Loss | ln(366) = 5.9026 |
| 实测初始 Loss | 5.9248（Train）/ 5.9210（Val） |
| 最佳 Val Loss | 3.8813（Iter ~400） |
| 最佳 Val PPL | 48.5 |
| 最终 Train Loss | 0.0690（PPL=1.1） |
| 最终 Val Loss | 4.4818（PPL=88.4） |
| 训练耗时 | 13.4s（GPU） |
| 吞吐量 | ~305,000 tok/s |
| Checkpoint 大小 | 9.84 MB（完整）/ 3.32 MB（仅模型） |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** Cell 4 报错 `ModuleNotFoundError: No module named 'torch'`

**应对：**
1. 在终端运行 `pip install torch`
2. 如果是 conda 环境：`conda install pytorch -c pytorch`
3. 重启 Kernel 后重新运行

### 场景 2：中文字体不显示

**症状：** Matplotlib 图表中中文显示为方框

**应对：**
1. 确认 `assets/fonts/NotoSansCJKsc-Regular.otf` 文件存在
2. 如果不存在，从 Google Fonts 下载 Noto Sans CJK SC
3. 或在 Cell 4 中修改字体路径指向系统中已有的中文字体

### 场景 3：训练时间过长

**症状：** CPU 上训练超过 10 分钟

**应对：**
1. 将 `max_iters` 从 2000 减少到 500
2. 将 `eval_interval` 从 200 减少到 100
3. 核心概念不受影响——仍然能观察到 loss 下降和过拟合

### 场景 4：数据文件找不到

**症状：** Cell 7 报错找不到数据文件

**应对：**
1. 确认 `data/` 目录在 notebook 所在目录或其父目录下
2. 检查 `resolve_data_dir()` 的候选路径是否包含实际数据位置
3. 可手动设置 `DATA_DIR = '/path/to/your/data'`

### 场景 5：CUDA 内存不足

**症状：** `RuntimeError: CUDA out of memory`

**应对：**
1. 减小 `batch_size`（从 32 改为 16 或 8）
2. 减小 `block_size`（从 64 改为 32）
3. 切换到 CPU：`device = 'cpu'`（本章模型很小，CPU 完全能跑）

### 场景 6：训练 Loss 不下降

**症状：** Loss 停在初始值附近不动

**应对：**
1. 检查学习率是否太小（应为 3e-4）
2. 检查数据是否正确加载（打印几个样本看看）
3. 确认 `model.train()` 被调用了
4. 检查 loss 反向传播后是否调用了 `optimizer.step()`